In [1]:
# Imports
import pandas as pd
from random import randint
from src import *
from src.simulator import SIMULATOR
import numpy as np

def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [2]:
# --------------------------------------------
#               INIT & CONFIG
# --------------------------------------------
sim = SIMULATOR()

DEBUG = 1
MAX_ITER = 300000

# DISCO-CGRA Parameters
nRCs = 4
nElementsPerVWRSlice = 32
nColsCGRA = 1
nElemsPerSPMLine = 128

In [3]:
# --------------------------------------------
#               KERNEL CONFIGURATION
# --------------------------------------------
kernel_path = './kernels/mmul_v1_tiling_32x32/'
kernel_number = 1 
column_usage = [True, True] 
nInstrPerCol = 45
imem_add_start = 0 
srf_spm_addres = 0 
version="_2col"

# Block size 32x32
BLOCK_SIZE = 32

sim.kernel_config(column_usage, nInstrPerCol, imem_add_start, srf_spm_addres, kernel_number)

In [ ]:
# --------------------------------------------
#               DATA
# --------------------------------------------
data = np.load(kernel_path + "data/data.npz")

A = data["A"]
B = data["B"]
C = data["C"]
expected_res = data["D"]

ROWS_A = int(data["ROWS_A"])
COLS_A = int(data["COLS_A"])
COLS_B = int(data["COLS_B"])

B_t = ((B.reshape(COLS_A, COLS_B)).T).flatten()
output = np.zeros((ROWS_A * COLS_B), dtype=np.int32)

print(f"Testing sizes A: {ROWS_A}x{COLS_A}, B: {COLS_A}x{COLS_B}, C: {ROWS_A}x{COLS_B}")


In [ ]:
# --------------------------------------------
#              COMPILE ASM TO HEX
# --------------------------------------------
sim.compileAsmToHex(kernel_path, kernel_number, version=version)

# --------------------------------------------
#          LOAD KERNEL INSTRUCTIONS
# --------------------------------------------

# This needs the hex instructions, if you don't provide them, generate then compiling the asm
sim.kernel_load(kernel_path, version=version + "_autogen", kernel_number=kernel_number)

In [6]:
# --------------------------------------------
#            SIMULATION PARAMETERS
# --------------------------------------------
show_lcu = []
show_srf = []
show_lsu = []
show_rcs = [[],[],[],[]]
show_mxcu = []
#display_ops = [show_lcu, show_lsu, show_mxcu, show_rcs, show_srf]
display_ops = [[] for _ in range(CGRA_ROWS + 4)]
# Default SRF values
srf = [0 for i in range(N_ELEMS_PER_VWR)]

In [7]:
# --------------------------------------------
#             FIT BLOCKING SIZE
# --------------------------------------------
nBlocksColsA = COLS_A // BLOCK_SIZE # Assuming divisible for now
options_rows_a_cols_b = [4, 8, 12, 16, 20, 24, 28, 32]
block_size_rowsA = 0
block_size_colsA = min(32, COLS_A)
block_size_colsB = 0
for i in range(len(options_rows_a_cols_b)):
    actA = False
    if ROWS_A >= options_rows_a_cols_b[i]:
        actA = True
        block_size_rowsA = options_rows_a_cols_b[i]
    
    actB = False
    if COLS_B >= options_rows_a_cols_b[i]:
        actB = True
        block_size_colsB = options_rows_a_cols_b[i]
    
    if (not actA) and (not actB):
        break


nLinesPerBlock = 8
nLinesBlockA = block_size_rowsA // 4
nLinesBlockB = block_size_colsB // 4

srf_spm_line = 0 # SRF 
spm_line_A = srf_spm_line + 1
spm_line_B = spm_line_A + nLinesBlockA
spm_line_C = spm_line_B + nLinesBlockB


In [ ]:
# --------------------------------------------
#             TILING LOOPS
# --------------------------------------------


nBlocksColsC = COLS_B // block_size_colsB
nBLocksRowsC = ROWS_A // block_size_rowsA


for blockCol in range(nBlocksColsC) :
    cC = blockCol * block_size_colsB
    for blockRow in range(nBLocksRowsC) :
        # Load C block
        rC = blockRow * block_size_rowsA
        aux_c_spm_line = spm_line_C
        c_row = rC
        for r in range(block_size_rowsA // 4):
            line = [0 for _ in range(nElemsPerSPMLine)]
            for rr in range(4):
                line[nElementsPerVWRSlice*rr : nElementsPerVWRSlice*rr + block_size_colsB] = C[(c_row + rr)*COLS_B:(c_row + rr)*COLS_B + block_size_colsB].copy()
            sim.setSPMLine(aux_c_spm_line, line.copy())
            print(f"Loading C on line {aux_c_spm_line}: {line}")
            aux_c_spm_line += 1
            c_row+=4

        # Output Stationary: Row A * Col B
        blockA = 0
        while blockA < nBlocksColsC :
            # Load A block
            aux_spm_line_A = spm_line_A
            a_row = rC
            for r in range(block_size_rowsA // 4):
                line = [0 for _ in range(nElemsPerSPMLine)]
                for rr in range(4):
                    line[nElementsPerVWRSlice*rr : nElementsPerVWRSlice*rr + block_size_colsA] = A[(a_row + rr)*COLS_A:(a_row + rr)*COLS_A + block_size_colsA].copy()
                sim.setSPMLine(aux_spm_line_A, line.copy())
                print(f"Loading A on line {aux_spm_line_A}: {line}")
                aux_spm_line_A += 1
                a_row+=4

            # Load B_t block
            aux_b_spm_line = spm_line_B
            b_t_row = 0
            for c in range(block_size_colsB // 4):
                line = [0 for _ in range(nElemsPerSPMLine)]
                for rr in range(4):
                    line[nElementsPerVWRSlice*rr : nElementsPerVWRSlice*rr + block_size_colsA] = B_t[(b_t_row + rr)*COLS_A:(b_t_row + rr)*COLS_A + block_size_colsA].copy()
                sim.setSPMLine(aux_b_spm_line, line.copy())
                print(f"Loading B on line {aux_b_spm_line}: {line}")
                aux_b_spm_line += 1
                b_t_row+=4

            # Default SRF values
            srf = [0 for i in range(N_ELEMS_PER_VWR)]

            nLinesPerRow = max((block_size_rowsA // 4)//2, (block_size_colsB // 4)//2)

            # Col 0
            srf[0]  = spm_line_A 
            srf[1]  = spm_line_B 
            srf[2]  = spm_line_C
            srf[3]  = (block_size_rowsA // 4)//2 -1 # ROWS_A / BLOCK_SIZE=4 / N_COLS=2
            srf[4]  = block_size_colsB // 4 -1 
            srf[5]  = block_size_colsA -1 
            srf[6]  = 0      # Not used
            srf[7]  = 0      # Not used
            # Col 1
            srf[8]  = spm_line_A + nLinesPerRow # Compute the next rows of C
            srf[9]  = spm_line_B
            srf[10] = spm_line_C + nLinesPerRow # Compute the next rows of C
            srf[11] = (block_size_rowsA // 4)//2 -1
            srf[12] = block_size_colsB // 4 -1 
            srf[13] = block_size_colsA -1 
            srf[14] = 0      # Not used
            srf[15] = 0      # Not used
            sim.setSPMLine(srf_spm_line, srf.copy())
            print(f"Loading SRF into line {srf_spm_line}: {srf}")

            # Compute 
            print("Run kernel")
            sim.run(kernel_number, display_ops=display_ops, max_iter=MAX_ITER)
            blockA += 1
        
        # Block C fully computed -> extract it
        
        cgra_out = []
        aux_c_spm_line = spm_line_C
        for _ in range(block_size_rowsA // 4):
            line = sim.getSPMLine(aux_c_spm_line)
            for rr in range(4):
                row_real = line[rr * nElementsPerVWRSlice : rr * nElementsPerVWRSlice + block_size_colsB]
                cgra_out.extend(c_int32(x).value for x in row_real)
            print(f"Extracting C line {aux_c_spm_line}: {line}")
            aux_c_spm_line += 1

        # Store block in output
        for r in range(block_size_rowsA):
            output[(rC+r) * COLS_B + cC : (rC+r) * COLS_B + cC + block_size_colsB] = cgra_out[r * block_size_colsB:(r + 1) * block_size_colsB].copy()
                

In [ ]:
# Verify results
disco_out = output
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != disco_out[i]:
        errors+=1
    
if errors == 0:
    print("The result is correct!")
else:
    print("Oops, something went wrong. There are " + str(errors) + " errors (out of " + str(len(expected_res)) + " elements).")
    print("DISCO out:")
    printAsMatrix(disco_out, ROWS_A, COLS_B)
    print("Expected result:")
    printAsMatrix(expected_res, ROWS_A, COLS_B)
    print("A:")
    printAsMatrix(A, ROWS_A, COLS_A)
    print("B_t:")
    printAsMatrix(B_t, COLS_B, COLS_A)
    print("C:")
    printAsMatrix(C, ROWS_A, COLS_B)